# Création de la base de données

## Les données utilisées

Dans le cadre de ce projet, l'objectif est d'évaluer l'efficacité réelle du Pass Culture. Pour ce faire, nous utilisons différentes bases de données.

In [1]:
import pandas as pd
from functions import *

## Basilic

la Base des lieux et équipements culturels (Basilic) est une base de données réalisée par agrégation de différentes sources : bases de la direction générale des patrimoines et de l’architecture, de la direction générale de la création artistique, de la direction générale des médias et des industries culturelles, de la délégation générale à la transmission, aux territoires et à la démocratie culturelle, du Centre national du cinéma et de l'image animée, du Centre national du livre, du Centre national des arts du cirque, de la rue et du théâtre (Artcena), de la Médiathèque du patrimoine et de la photographie.

Elle porte sur le champ de la France entière, et va nous permettre de trouver toutes les infrastructures culturelles.

In [2]:
# Importation des données

importdata("basilic_29072026.csv", ";")


FileNotFoundError: [Errno 2] No such file or directory: 'basilic_29072026.csv'

Voici maintenant les informations générales sur la base, préalables au nettoyage.

In [ ]:
# Informations générales sur la base

infosbase(df_brut_basilic)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 86366
Nombre de colonnes  : 54

NOMS DES COLONNES ET TYPES
Nom                                         object
Adresse                                     object
Complement Adresse                         float64
Code Postal                                 object
libelle_geographique                        object
code_insee                                  object
Code Insee Arrondt                          object
Identifiant origine                         object
Type équipement ou lieu                     object
Label et appellation                        object
Région                                      object
Domaine                                     object
Archéologie détail                          object
Adresse postale                             object
Département                                 object
Précision équipement                        object
N_Département                               object
N_Région               

In [6]:
print("\n" + "=" * 60)
print("VALEURS MANQUANTES PAR COLONNE")
print("=" * 60)
print(df_brut_basilic.isna().sum())


VALEURS MANQUANTES PAR COLONNE
Nom                                            0
Adresse                                    31397
Complement Adresse                         86366
Code Postal                                    9
libelle_geographique                           0
code_insee                                     0
Code Insee Arrondt                            85
Identifiant origine                         3013
Type équipement ou lieu                        0
Label et appellation                       30483
Région                                        12
Domaine                                        0
Archéologie détail                         85804
Adresse postale                                0
Département                                   12
Précision équipement                       65851
N_Département                                  9
N_Région                                      12
Fonction_1                                  7372
Fonction_2                           

#### Choix des variables

Maintenant, nous allons nettoyer la base, pour ne garder que les variables qui sont pertinentes pour notre sujet.

Ainsi, nous allons garder quelques variables permettant d'identifier l'infrastructure de manière unique ("Nom", "libelle_geographique", "code_insee", "Code Insee Arrondt", "Label et appellation", "Région", "Département", "Fonction_1", "Fonction_2", "Fonction_3", "Fonction_4", "Type_de_cinema", "Nombre_ecrans", "Nombre_de_salles_de_theatre", "Surface_Bibliotheque", "Precision_protection_sites_et_monuments").

In [7]:
df_basilic = df_brut_basilic[["Nom", 
    "libelle_geographique",
    "Département",
    "Région",  
    "code_insee", 
    "Code Insee Arrondt", 
    "Label et appellation", 
    "Fonction_1", 
    "Fonction_2", 
    "Fonction_3", 
    "Fonction_4", 
    "Type_de_cinema", 
    "Nombre_ecrans", 
    "Nombre_de_salles_de_theatre", 
    "Surface_Bibliotheque", 
    "Precision_protection_sites_et_monuments"]]

Nous allons maintenant transformer la base pour que la granularité ne soit pas au niveau du batiment, mais au niveau de la ville : nous aurons ainsi une ligne par commune, et pour chaque type d'équipement, le nombre d'établissements, et le cas échéant, le nombre total de salles ou la surface totale (cinéma, théâtre, bibliothèque...).

La première étape est de vérifier la colonne à partir de laquelle nous allons compter le nombre de bâtiments.

In [8]:
print("Label et appellation", df_basilic["Label et appellation"].unique())

Label et appellation ['Monument historique' nan 'Centre d’art contemporain d’intérêt national'
 'Centre chorégraphique national'
 'Centre de développement chorégraphique national'
 'Centre dramatique national' 'Art et essai'
 'Centre culturel de rencontre' 'Scène conventionnée d’intérêt national'
 'Scène nationale' 'Scène de musiques actuelles'
 'Site patrimonial remarquable' 'Théâtre hors label' 'Théâtre national'
 'Théâtre privé' 'Théâtre de ville' "Patrimoine mondial de l'Unesco"
 'Microfolie' 'Centre national des arts de la rue et de l’espace public'
 'Centre national de création musicale' 'Compagnie avec lieu'
 'Compagnie subventionnée (DRAC - Aide à la production)'
 'Compagnie conventionnée' "Fonds régional d'art contemporain"
 'Itinéraire culturel européen' 'Jardin remarquable'
 'LIR - Librairie indépendante de référence' 'LR - Librairie de référence'
 'Maison des illustres' 'Monument national' 'Musée de France'
 'Orchestre national en région' 'Pôle national du cirque'
 'Archite

Maintenant, nous pouvons compter le nombre de bâtiment par ville, pour chaque type et au total.

In [9]:
# Indicatrices, avec les NaN regroupés dans "autre_etablissements"
# ici on commence par vérifier qu'on n'a pas déjà une colonne autre_etablissement
assert "autre_etablissements" not in df_basilic["Label et appellation"].unique()
label_dummies = pd.get_dummies(
    df_basilic["Label et appellation"].fillna("autre_etablissements")
)

# Agrégation par commune
agg_labels = label_dummies.groupby(df_basilic["code_insee"]).sum()

# Agrégats numériques
agg_numeriques = df_basilic.groupby("code_insee").agg(
    nb_ecrans_total=("Nombre_ecrans", "sum"),
    nb_salles_theatre_total=("Nombre_de_salles_de_theatre", "sum"),
    surface_bibliotheque_total=("Surface_Bibliotheque", "sum"),
    nb_etablissements=("Nom", "count")
)

# Infos communales
group_keys = ["code_insee", "libelle_geographique", "Département", "Région"]
infos_commune = df_basilic[group_keys].drop_duplicates(subset="code_insee").set_index("code_insee")

# Fusion finale
df_communes = infos_commune.join([agg_labels, agg_numeriques]).reset_index()

In [10]:
# Informations sur la base de données

infosbase(df_communes)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 22084
Nombre de colonnes  : 49

NOMS DES COLONNES ET TYPES
code_insee                                                   object
libelle_geographique                                         object
Département                                                  object
Région                                                       object
Architecture contemporaine remarquable                        int64
Art et essai                                                  int64
Centre chorégraphique national                                int64
Centre culturel de rencontre                                  int64
Centre de développement chorégraphique national               int64
Centre dramatique national                                    int64
Centre d’art contemporain d’intérêt national                  int64
Centre national de création musicale                          int64
Centre national de la marionnette                             int64
Centre

A ce stade, la base de données recense donc l'ensemble des équipements disponibles pour chaque commune. Nous allons maintenant y ajouter des informations complémentaires telles que des informations démographiques.

## Démographie et âges (INSEE)

Nous importons maintenant la base de données de l'INSEE permettant d'obtenir la population des communes par tranche d'âge de cinq ans et par sexe. Le but sera de joindre cette base avec les données issues de la base Basilic afin d'arriver à obtenir la densité d'équipements culturels pas habitant.

Dans un premier temps, nous importons les données. Comme pour la base Basilic, elles ont été téléchargées (le 30 juillet 2026) et stockées localement.

In [15]:
# Importation des données
path_data_pop = "DS_RP_TD_POPULATION_AGEHARSEX_PRINC_2023_data.csv"

# Attention pour cette base le séparateur est un ;
df_brut_pop = pd.read_csv(path_data_pop, sep=";", encoding="utf-8")



/tmp/ipykernel_49676/1775215519.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_brut_pop = pd.read_csv(path_data_pop, sep=";", encoding="utf-8")


Il s'agit ensuite de regarder les modalités des variables afin de sélectionner seulement les lignes qui nous intéressent.

In [16]:
# Informations sur la base de données

infosbase(df_brut_pop)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 27567540
Nombre de colonnes  : 10

NOMS DES COLONNES ET TYPES
GEO             object
GEO_OBJECT      object
RP_MEASURE      object
AGE             object
HAR             object
SEX             object
FREQ            object
OBS_STATUS      object
TIME_PERIOD      int64
OBS_VALUE      float64
dtype: object


In [20]:
print("GEO_OBJECT", df_brut_pop["GEO_OBJECT"].unique())

GEO_OBJECT ['AAV2020' 'ARM' 'ARR' 'BV2022' 'COM' 'DEP' 'EPCI' 'FRANCE' 'REG' 'UU2020'
 'ZE2020']


In [23]:
print("AGE", df_brut_pop["AGE"].unique())

AGE ['Y0T4' 'Y10T14' 'Y15T19' 'Y20T24' 'Y25T29' 'Y30T34' 'Y35T39' 'Y40T44'
 'Y45T49' 'Y50T54' 'Y55T59' 'Y5T9' 'Y60T64' 'Y65T69' 'Y70T74' 'Y75T79'
 'Y80T84' 'Y85T89' 'Y90T94' 'Y95T99' 'Y_GE100' '_T']


In [22]:
df_pop = df_brut_pop[["GEO",
    "GEO_OBJECT",
    "AGE",
    "HAR",
    "SEX",
    "FREQ",
    "OBS_STATUS",
    "OBS_VALUE"]]

In [25]:
df_pop = df_pop[
    (df_pop['AGE'].isin(['Y15T19', 'Y20T24'])) & (df_pop['GEO_OBJECT'] == 'COM')
]

## Revenus et niveau de vie (INSEE - Filosofi)

Nous allons maintenant importer les données de revenu et de niveau de vie qui pourront servir de variables de contrôle ou d'instrument dans l'analyse de l'influence du Pass Culture sur les pratiques culturelles des jeunes.

In [26]:
# Importation des données

path_data_filosofi = "DS_FILOSOFI_CC_2023_data.csv"

# Attention pour cette base le séparateur est un ;
df_brut_filosofi = pd.read_csv(path_data_filosofi, sep=";", encoding="utf-8")

/tmp/ipykernel_49676/3120185984.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_brut_filosofi = pd.read_csv(path_data_filosofi, sep=";", encoding="utf-8")


In [27]:
infosbase(df_brut_filosofi)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1123173
Nombre de colonnes  : 9

NOMS DES COLONNES ET TYPES
FILOSOFI_MEASURE     object
GEO                  object
GEO_OBJECT           object
UNIT_MEASURE         object
CONF_STATUS          object
OBS_STATUS           object
UNIT_MULT             int64
TIME_PERIOD           int64
OBS_VALUE           float64
dtype: object
